# 2.0 Validate Accessibility Metrics

Inspect per-school metrics and administrative aggregations from Phase 2.

**Checks:**
1. Metric distributions (isolation score, nearest private, desert flags)
2. Known-area spot checks (NCR should have low isolation, CAR/BARMM should have high)
3. Private desert and ESC desert geographic patterns
4. Municipal aggregation sanity (school counts, coverage)
5. Most isolated schools — do they make geographic sense?

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

from modules.gcs_utils import OUTPUT_DIR

METRICS_DIR = OUTPUT_DIR / "metrics"
AGG_DIR = OUTPUT_DIR / "aggregations"

df = pd.read_parquet(METRICS_DIR / "school_accessibility.parquet")
df_muni = pd.read_parquet(AGG_DIR / "municipal_summary.parquet")
df_prov = pd.read_parquet(AGG_DIR / "provincial_summary.parquet")
df_reg = pd.read_parquet(AGG_DIR / "regional_summary.parquet")

print(f"School metrics: {len(df):,} rows, {len(df.columns)} columns")
print(f"Municipal: {len(df_muni):,} | Provincial: {len(df_prov):,} | Regional: {len(df_reg):,}")
print(f"\nColumns: {list(df.columns)}")

School metrics: 56,018 rows, 33 columns
Municipal: 1,752 | Provincial: 139 | Regional: 18

Columns: ['school_id', 'school_name', 'latitude', 'longitude', 'region', 'province', 'municipality', 'sector', 'offers_es', 'offers_jhs', 'offers_shs', 'urban_rural', 'enrollment_status', 'osrm_status', 'island_group', 'n_neighbors_5km', 'n_public_5km', 'n_private_5km', 'n_neighbors_10km', 'n_public_10km', 'n_private_10km', 'n_neighbors_20km', 'n_public_20km', 'n_private_20km', 'nearest_private_km', 'nearest_private_id', 'nearest_public_km', 'nearest_esc_km', 'nearest_esc_id', 'mean_road_haversine_ratio', 'private_desert', 'esc_desert', 'isolation_score']


## 1. Metric distributions

In [2]:
# Key metric distributions
print("ISOLATION SCORE")
print(df["isolation_score"].describe().to_string())

print("\nNEAREST PRIVATE SCHOOL (km)")
nearest_prv = df["nearest_private_km"].replace(np.inf, np.nan)
print(nearest_prv.describe().to_string())

print("\nNEAREST ESC SCHOOL (km)")
nearest_esc = df["nearest_esc_km"].replace(np.inf, np.nan)
print(nearest_esc.describe().to_string())

print("\nNEIGHBOR COUNTS")
for col in ["n_neighbors_5km", "n_neighbors_10km", "n_neighbors_20km"]:
    print(f"  {col}: mean={df[col].mean():.1f}, median={df[col].median():.0f}, max={df[col].max()}")

print("\nDESERT FLAGS")
print(f"  Private desert (no private within 10km): {df['private_desert'].sum():,} ({df['private_desert'].mean()*100:.1f}%)")
print(f"  ESC desert (no ESC within 10km): {df['esc_desert'].sum():,} ({df['esc_desert'].mean()*100:.1f}%)")

print("\nROAD DETOUR FACTOR")
ratio = df["mean_road_haversine_ratio"].dropna()
print(f"  mean={ratio.mean():.2f}x, median={ratio.median():.2f}x, p95={ratio.quantile(0.95):.2f}x")

ISOLATION SCORE
count    56018.000000
mean         0.118618
std          0.153179
min          0.002975
25%          0.034005
50%          0.070697
75%          0.140103
max          1.000000

NEAREST PRIVATE SCHOOL (km)
count    48723.000000
mean         5.522385
std          5.142199
min          0.000700
25%          1.084550
50%          3.992800
75%          8.541800
max         19.999802

NEAREST ESC SCHOOL (km)
count    45134.000000
mean         6.517943
std          5.363319
min          0.000700
25%          1.753125
50%          5.204350
75%         10.191700
max         19.998501

NEIGHBOR COUNTS
  n_neighbors_5km: mean=17.3, median=8, max=212
  n_neighbors_10km: mean=54.9, median=26, max=634
  n_neighbors_20km: mean=176.4, median=87, max=1869

DESERT FLAGS
  Private desert (no private within 10km): 16,943 (30.2%)
  ESC desert (no ESC within 10km): 22,495 (40.2%)

ROAD DETOUR FACTOR
  mean=1.53x, median=1.49x, p95=2.05x


## 2. Regional comparison (sanity check)

NCR should have the lowest isolation scores. CAR, BARMM, MIMAROPA should be highest.

In [3]:
print("REGIONAL METRICS (sorted by mean isolation score)")
print("=" * 100)
cols = ["region", "n_schools", "n_public", "n_private", "mean_isolation_score",
        "pct_private_desert", "pct_esc_desert", "mean_nearest_private_km"]
print(df_reg[cols].sort_values("mean_isolation_score", ascending=False).to_string(index=False))

REGIONAL METRICS (sorted by mean isolation score)
     region  n_schools  n_public  n_private  mean_isolation_score  pct_private_desert  pct_esc_desert  mean_nearest_private_km
 Region XII       2381      2070        311              0.291177                33.2            47.4                 6.815031
        CAR       1994      1843        151              0.173222                56.0            63.2                 8.788244
Region VIII       4326      4168        158              0.172851                55.4            67.5                 8.514250
   MIMAROPA       2587      2362        225              0.160918                49.5            61.2                 7.482037
  Region XI       2507      2195        312              0.159608                44.9            55.2                 7.623067
     CARAGA       2293      2094        199              0.147202                44.2            61.3                 7.666403
   Region X       2853      2498        355              0.14

## 3. Private desert and ESC desert by region

In [4]:
# Desert flags by region (public schools only)
df_pub = df[df["sector"] == "public"]
desert = df_pub.groupby("region").agg(
    n_public=("school_id", "count"),
    n_private_desert=("private_desert", "sum"),
    pct_private_desert=("private_desert", "mean"),
    n_esc_desert=("esc_desert", "sum"),
    pct_esc_desert=("esc_desert", "mean"),
).sort_values("pct_private_desert", ascending=False)

desert["pct_private_desert"] = (desert["pct_private_desert"] * 100).round(1)
desert["pct_esc_desert"] = (desert["pct_esc_desert"] * 100).round(1)
desert["n_private_desert"] = desert["n_private_desert"].astype(int)
desert["n_esc_desert"] = desert["n_esc_desert"].astype(int)

print("PUBLIC SCHOOLS: PRIVATE AND ESC DESERTS BY REGION")
print("=" * 85)
print(desert.to_string())

PUBLIC SCHOOLS: PRIVATE AND ESC DESERTS BY REGION
             n_public  n_private_desert  pct_private_desert  n_esc_desert  pct_esc_desert
region                                                                                   
CAR              1843              1033                56.0          1165            63.2
Region VIII      4168              2311                55.4          2812            67.5
BARMM            2486              1358                54.6          1595            64.2
MIMAROPA         2362              1170                49.5          1446            61.2
Region V         3856              1742                45.2          2246            58.2
Region XI        2195               986                44.9          1212            55.2
CARAGA           2094               926                44.2          1283            61.3
Region IX        2542              1114                43.8          1587            62.4
NIR              2201               793           

## 4. Most isolated schools

Top 20 schools by isolation score — verify these are in remote, mountainous, or island areas.

In [5]:
# Most isolated schools (only those with computed OSRM status — genuine isolation)
genuine = df[df["osrm_status"] == "computed"].nlargest(20, "isolation_score")
cols = ["school_id", "school_name", "region", "municipality", "sector",
        "isolation_score", "n_neighbors_5km", "n_neighbors_20km",
        "nearest_private_km", "private_desert", "latitude", "longitude"]
print("TOP 20 MOST ISOLATED SCHOOLS (osrm_status=computed)")
print("=" * 140)
print(genuine[cols].to_string(index=False))

TOP 20 MOST ISOLATED SCHOOLS (osrm_status=computed)
school_id                   school_name      region municipality sector  isolation_score  n_neighbors_5km  n_neighbors_20km  nearest_private_km  private_desert  latitude  longitude
   103534    Alomanay Elementary School   Region II      PALANAN public              1.0                0                 0                 inf            True 17.092270 122.389713
   103536          Bisag Primary School   Region II      PALANAN public              1.0                0                 0                 inf            True 17.012530 122.382488
   103539   Dialawyao Elementary School   Region II      PALANAN public              1.0                0                 0                 inf            True 17.087058 122.438196
   103542       Dibungko Primary School   Region II      PALANAN public              1.0                0                 0                 inf            True 17.069323 122.445102
   103543   Dibutarek Elementary School   R

## 5. Municipal aggregation sanity

In [6]:
# Municipal summary overview
print(f"Municipal units: {len(df_muni):,}")
print(f"  Total schools accounted: {df_muni['n_schools'].sum():,}")
print(f"  Mean schools/municipality: {df_muni['n_schools'].mean():.1f}")
print(f"  Municipalities with 100% private desert: {(df_muni['pct_private_desert'] == 100).sum():,}")
print(f"  Municipalities with 100% ESC desert: {(df_muni['pct_esc_desert'] == 100).sum():,}")

# Worst municipalities by private desert
print("\nTOP 10 MUNICIPALITIES BY PRIVATE DESERT % (min 10 public schools)")
worst = df_muni[df_muni["n_public"] >= 10].nlargest(10, "pct_private_desert")
cols = ["region", "province", "municipality", "n_public", "n_private",
        "pct_private_desert", "mean_nearest_private_km", "mean_isolation_score"]
print(worst[cols].to_string(index=False))

# Best municipalities (lowest isolation)
print("\nTOP 10 MOST CONNECTED MUNICIPALITIES (min 10 public schools)")
best = df_muni[df_muni["n_public"] >= 10].nsmallest(10, "mean_isolation_score")
print(best[cols].to_string(index=False))

Municipal units: 1,752
  Total schools accounted: 55,577
  Mean schools/municipality: 31.7
  Municipalities with 100% private desert: 224
  Municipalities with 100% ESC desert: 338

TOP 10 MUNICIPALITIES BY PRIVATE DESERT % (min 10 public schools)
region      province          municipality  n_public  n_private  pct_private_desert  mean_nearest_private_km  mean_isolation_score
 BARMM       BASILAN   HADJI MOHAMMAD AJUL        12          0               100.0                16.762421              0.126022
 BARMM       BASILAN           TABUAN-LASA        15          0               100.0                10.394950              0.049374
 BARMM       BASILAN             TIPO-TIPO        14          0               100.0                17.346878              0.095770
 BARMM       BASILAN         UNGKAYA PUKAN        13          0               100.0                15.928434              0.110301
 BARMM LANAO DEL SUR             BINIDAYAN        17          0               100.0              

## 6. Sector comparison

In [ ]:
# Compare public vs private school accessibility
for sector in ["public", "private"]:
    s = df[df["sector"] == sector]
    print(f"\n{sector.upper()} SCHOOLS ({len(s):,})")
    print(f"  Isolation score: mean={s['isolation_score'].mean():.4f}, median={s['isolation_score'].median():.4f}")
    print(f"  Neighbors (5km): mean={s['n_neighbors_5km'].mean():.1f}, median={s['n_neighbors_5km'].median():.0f}")
    print(f"  Neighbors (20km): mean={s['n_neighbors_20km'].mean():.1f}, median={s['n_neighbors_20km'].median():.0f}")
    prv_km = s["nearest_private_km"].replace(np.inf, np.nan)
    esc_km = s["nearest_esc_km"].replace(np.inf, np.nan)
    print(f"  Nearest private: mean={prv_km.mean():.1f} km, median={prv_km.median():.1f} km")
    print(f"  Nearest ESC: mean={esc_km.mean():.1f} km, median={esc_km.median():.1f} km")
    print(f"  Private desert: {s['private_desert'].sum():,} ({s['private_desert'].mean()*100:.1f}%)")
    print(f"  ESC desert: {s['esc_desert'].sum():,} ({s['esc_desert'].mean()*100:.1f}%)")